# 12 — RQ2 clean comments

Cleans the raw RQ2 corpus with the same procedure as RQ1 (notebook 02): drop duplicate comment IDs, HTML-decode and normalise whitespace, remove blanks, and keep English comments. RQ2-specific rules: comments are kept at **≥10 words**, and text is **not** de-duplicated — identical comments with different IDs are legitimate separate observations for prevalence estimation. The corpus is then **decontaminated** against the RQ1 labelled set so RQ2 evaluation never overlaps RQ1 training.

Reads `data/raw/rq2/rq2_raw_comments.csv`; writes the cleaned, decontaminated corpus to `data/processed/`.

In [ ]:
import html
from pathlib import Path

import pandas as pd
from langdetect import detect, DetectorFactory, LangDetectException

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE       = PROJECT_ROOT / "data" / "raw" / "rq2" / "rq2_raw_comments.csv"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_FILE     = PROCESSED_DIR / "rq2_clean.csv"
FINAL_FILE     = PROCESSED_DIR / "rq2_primary_12mo_decontaminated.csv"

df = pd.read_csv(RAW_FILE)
print("Raw:", df.shape)


In [ ]:
# 1. Drop duplicate comment IDs
before = len(df)
clean = df.drop_duplicates(subset="comment_id", keep="first").reset_index(drop=True)
print(f"{before:,} -> {len(clean):,} after duplicate comment-ID removal")

# 2. Text normalisation (same as RQ1): HTML-decode, collapse whitespace, strip
clean["text_clean"] = (
    clean["text"].map(lambda x: html.unescape(str(x)))
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip()
)
before = len(clean)
clean = clean[(clean["text_clean"].str.len() > 0)
              & (clean["text_clean"].str.lower() != "nan")].reset_index(drop=True)
print(f"{before:,} -> {len(clean):,} after removing blank text")

# NOTE: text is deliberately NOT de-duplicated for RQ2 (prevalence estimation).

# 3. Word-count rule for RQ2: keep >= 10 words
clean["n_words"] = clean["text_clean"].str.split().str.len()
before = len(clean)
clean = clean[clean["n_words"] >= 10].reset_index(drop=True)
print(f"{before:,} -> {len(clean):,} after keeping >=10 words")

# 4. English-language filter (deterministic)
DetectorFactory.seed = 0
def is_english(t):
    try:
        return detect(t) == "en"
    except LangDetectException:
        return False
before = len(clean)
clean = clean[clean["text_clean"].map(is_english)].reset_index(drop=True)
print(f"{before:,} -> {len(clean):,} after English filter")

clean.to_csv(CLEAN_FILE, index=False, encoding="utf-8-sig")
print("Communities:", clean["community_group"].nunique(),
      "| channels:", clean["channel_name"].nunique())


In [ ]:
# 5. Decontaminate against the RQ1 labelled set
# Remove any RQ2 comment whose (normalised) text also appears in the RQ1 labelled
# corpus, so RQ2 evaluation never overlaps the data used to build the classifiers.
RQ1_PATH = PROCESSED_DIR / "labelled_comments_final.csv"
if not RQ1_PATH.exists():
    raise FileNotFoundError(
        "Could not find data/processed/labelled_comments_final.csv.\n"
        "Place the RQ1 labelled file there before decontaminating.")

rq1 = pd.read_csv(RQ1_PATH)

def norm(s):
    return " ".join(str(s).lower().split())

rq1_texts = set(rq1["text_clean"].map(norm))
before = len(clean)
overlap_mask = clean["text_clean"].map(norm).isin(rq1_texts)
decontaminated = clean[~overlap_mask].reset_index(drop=True)
print(f"{before:,} -> {len(decontaminated):,} after removing "
      f"{int(overlap_mask.sum())} RQ1-overlapping comments")

decontaminated.to_csv(FINAL_FILE, index=False, encoding="utf-8-sig")
print("\nFinal decontaminated RQ2 corpus:", decontaminated.shape)
print(decontaminated.groupby("community_group")["channel_name"].nunique())
print("Saved:", FINAL_FILE)
